# Deploy a LangGraph Agent with the Mosaic AI Agent Framework (v2)

This is the **same CS4603 study-assistant agent** you deployed in
`15.databricks_deployment/`, but deployed the **modern** way — with the
Databricks Agent Framework and `databricks.agents.deploy()`.

**Run this notebook inside your Databricks workspace** (not locally).

## What's different from v1?

| | v1 (`15.databricks_deployment`) | v2 (this notebook) | Benefit of v2 |
|---|---|---|---|
| Model file | `agent.py` — bare LangGraph `MessagesState` | `agent_chat.py` — wrapped as `ChatAgent` | Standard agent signature works with AI Playground, evaluation, and the review app out of the box |
| LLM client | `ChatOpenAI` + `DATABRICKS_TOKEN` | `ChatDatabricks` (no token) | No credential lives in your code — nothing to leak or rotate |
| Log API | `mlflow.langchain.log_model()` | `mlflow.pyfunc.log_model(resources=[...])` | Declaring `resources` is what unlocks automatic auth and dependency tracking |
| Deploy API | `w.serving_endpoints.create(...)` | `agents.deploy(...)` | One call instead of hand-wiring endpoint config and secret env vars |
| **Auth** | **manual** — `cs4603-deploy` secret scope | **automatic** — via logged `resources` | No secret scope to create, inject, or maintain; credentials are short-lived and least-privilege |
| Extras | endpoint only | endpoint **+ review app + eval + monitoring** | Human feedback, inference tables, and tracing for free — no extra setup |

The star of the show is **automatic authentication** — you no longer create a
secret scope or inject `DATABRICKS_TOKEN`. Read Step 2 to see how.

## Step 0 — Prerequisites

1. Upload `agent_chat.py` to your Databricks workspace (same folder as this notebook).
2. Access to Unity Catalog (`main.default` catalog/schema).
3. Model Serving enabled in your workspace.

Note: unlike v1, you do **not** need to set up the `cs4603-deploy` secret scope.

## Step 1 — Install Dependencies

`databricks-agents` is the new one here — it provides `agents.deploy()`.
`databricks-langchain` provides `ChatDatabricks`.

In [ ]:
%pip install -U -qqqq mlflow databricks-agents databricks-langchain "langgraph<0.4" langchain-core langchain
dbutils.library.restartPython()


## Step 2 — Verify `agent_chat.py` imports

Before logging, make sure the model file loads. Because it uses
`ChatDatabricks`, it authenticates automatically inside the workspace — there
are **no environment variables to set** (contrast this with v1's Step 2, where
you had to export `DATABRICKS_HOST/TOKEN/MODEL`).

In [ ]:
import importlib.util
import mlflow

# Suppress set_model during the test import
_orig = mlflow.models.set_model
mlflow.models.set_model = lambda *a, **kw: None

spec = importlib.util.spec_from_file_location("agent_chat", "agent_chat.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

mlflow.models.set_model = _orig

print(f"Loaded agent with {len(mod.tools)} tools")
print(f"LLM endpoint: {mod.LLM_ENDPOINT_NAME}")

# Quick local smoke test — note: no token needed, ChatDatabricks auto-authenticates
result = mod.AGENT.predict(
    {"messages": [{"role": "user", "content": "Convert 100 F to Celsius"}]}
)
print(result.messages[-1].content)


## Step 3 — Log the model with `resources` (this is where auto-auth is set up)

We log with `mlflow.pyfunc.log_model()` and pass a **`resources`** list naming
every Databricks resource the agent needs at runtime — here, the LLM serving
endpoint.

**This one argument is what enables automatic authentication.** When Databricks
deploys the model, it reads the declared resources and provisions a
short-lived, least-privilege credential so the agent can call
`databricks-qwen35-122b-a10b` — no PAT, no secret scope. (If your agent also
used a Vector Search index or a UC function, you'd add
`DatabricksVectorSearchIndex(...)` / `DatabricksFunction(...)` here too.)

In [ ]:
import mlflow
from importlib.metadata import version as _v
from mlflow.models.resources import DatabricksServingEndpoint

LLM_ENDPOINT_NAME = "databricks-qwen35-122b-a10b"

mlflow.set_registry_uri("databricks-uc")
experiment_name = f"/Users/{spark.sql('SELECT current_user()').first()[0]}/wk5-deployment-v2"
mlflow.set_experiment(experiment_name)

# Pin the SERVING env to the EXACT versions we validated above. If serving used
# unpinned deps it could pull a too-new langgraph and fail at deploy time with
# the same "Please install langchain>=0.2.17 and langgraph>=0.2.0" error.
pip_requirements = [
    f"mlflow=={_v('mlflow')}",
    f"databricks-langchain=={_v('databricks-langchain')}",
    f"databricks-agents=={_v('databricks-agents')}",
    f"langgraph=={_v('langgraph')}",
    f"langchain-core=={_v('langchain-core')}",
    f"langchain=={_v('langchain')}",
]

with mlflow.start_run(run_name="agent-framework") as run:
    logged = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent_chat.py",  # models-from-code, same idea as v1
        resources=[
            # ← the line that grants automatic auth to the LLM endpoint
            DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME),
        ],
        input_example={"messages": [{"role": "user", "content": "What is RAG?"}]},
        pip_requirements=pip_requirements,
    )

print(f"Model URI: {logged.model_uri}")
print(f"Run ID:    {run.info.run_id}")


## Step 4 — Register in Unity Catalog

Same as v1 — but note the **new model name** (`..._v2`) so this does not clobber
the v1 model.

In [ ]:
UC_MODEL_NAME = "main.default.cs4603_agent_v2"

uc_model = mlflow.register_model(model_uri=logged.model_uri, name=UC_MODEL_NAME)
print(f"Registered: {UC_MODEL_NAME} version {uc_model.version}")

## Step 5 — Deploy with `agents.deploy()`

This is the big simplification. In v1 you called
`w.serving_endpoints.create(...)` and hand-wired `environment_vars` with secret
references. Here, **one call** does all of it:

- creates (or updates) the serving endpoint,
- wires up the automatic credentials from the `resources` you logged,
- stands up a **review app** for human feedback,
- enables inference tables + tracing for evaluation and monitoring.

First deploy takes **3–8 minutes**.

In [ ]:
from databricks import agents

deployment = agents.deploy(
    UC_MODEL_NAME,
    uc_model.version,
    scale_to_zero=True,
)

print(f"Endpoint name: {deployment.endpoint_name}")
print(f"Query URL:     {deployment.query_endpoint}")
print(f"Review app:    {deployment.review_app_url}")
print("\nDeployment takes 3-8 minutes. Watch the Serving UI for READY.")

## Step 6 — Test the deployed endpoint

Once **READY**, call it with the same `openai.OpenAI` pattern used across the
course. 

In [ ]:
import openai

UC_MODEL_NAME = "main.default.cs4603_agent_v2"
try:
    endpoint_name = deployment.endpoint_name
except NameError:
    from databricks import agents

    deployments = agents.get_deployments(UC_MODEL_NAME)
    if not deployments:
        raise RuntimeError(
            f"No active deployment found for {UC_MODEL_NAME}. Run Step 5 first."
        )
    deployment = deployments[0]
    endpoint_name = deployment.endpoint_name

print(f"Using endpoint: {endpoint_name}")

workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = (
    dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
)

client = openai.OpenAI(
    api_key=token,
    base_url=f"https://{workspace_url}/serving-endpoints",
)

response = client.chat.completions.create(
    model=endpoint_name,
    messages=[{"role": "user", "content": "Convert 100 degrees Fahrenheit to Celsius"}],
)

print(response.choices[0].message.content)

## Cleanup

To delete the endpoint and stop billing:

```python
from databricks import agents
agents.delete_deployment("main.default.cs4603_agent_v2")
```

With `scale_to_zero=True`, the endpoint costs nothing when idle — only delete it
if you want to fully remove the resource.